In [ ]:
from __future__ import annotations

import itertools
from dataclasses import asdict
from typing import Iterable, Sequence

import numpy as np
import pandas as pd

from .metrics import compute_metrics
from .model import PCSSOR, PCSSORParams


def default_fast_grid() -> dict[str, Sequence[float | int]]:
    return {
        "K": [7, 9, 11],
        "M": [21, 25, 29],
        "sigma_scale": [1.0, 1.5, 2.0],
        "lam": [1e-8, 1e-6, 1e-4],
        "eta": [0.0, 1e-3],
        "mu_sym": [0.0, 10.0, 50.0],
    }


def default_full_grid() -> dict[str, Sequence[float | int]]:
    return {
        "K": [5, 7, 9, 11, 13],
        "M": [17, 21, 25, 29, 33],
        "sigma_scale": [0.8, 1.0, 1.5, 2.0, 2.5],
        "lam": [1e-10, 1e-8, 1e-6, 1e-4, 1e-2],
        "eta": [0.0, 1e-4, 1e-3, 1e-2],
        "mu_sym": [0.0, 10.0, 50.0, 200.0],
    }


def iter_param_grid(grid: dict) -> Iterable[PCSSORParams]:
    keys = list(grid.keys())
    for values in itertools.product(*[grid[k] for k in keys]):
        kwargs = dict(zip(keys, values))
        yield PCSSORParams(**kwargs)


def tune_pcssor(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    grid: dict | None = None,
    criterion: str = "RMSE",
) -> tuple[PCSSOR, dict, pd.DataFrame]:
    """Grid-search PC-SSOR hyperparameters on the validation set."""
    grid = grid or default_fast_grid()
    rows = []
    best_model = None
    best_metrics = None
    best_score = None

    for trial_id, params in enumerate(iter_param_grid(grid), start=1):
        model = PCSSOR(params).fit(train_df)
        pred = model.predict(valid_df)
        metrics = compute_metrics(valid_df["Cp"].values, pred)
        row = {"trial": trial_id, **asdict(params), **{f"valid_{k}": v for k, v in metrics.items()}}
        rows.append(row)
        score = metrics[criterion]
        if best_score is None or score < best_score:
            best_score = score
            best_model = model
            best_metrics = metrics

    return best_model, best_metrics, pd.DataFrame(rows)


def fit_final_model(train_df: pd.DataFrame, valid_df: pd.DataFrame, params: PCSSORParams) -> PCSSOR:
    train_valid = pd.concat([train_df, valid_df], ignore_index=True)
    return PCSSOR(params).fit(train_valid)
